In [2]:
import lightgbm as lgb
import shap
import numpy as np
import pandas as pd

from load_data import load_data_with_folds, get_fold_data

X, y, patient_ids, fold_indices = load_data_with_folds(
    file_path="../data/slice_localization_data.csv",
    target_col="reference",
    drop_cols=["patientId"],
    n_folds=10,
    random_state=42
)

X_train, y_train, X_test, y_test = get_fold_data(
    X, y, patient_ids, fold_indices, 0
)

if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train)
    X_test = pd.DataFrame(X_test)


/Users/larsabbink/Documents/persoonlijk/LightGBM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
params = {
    'objective': 'regression',
    'boosting_type': 'dart',
    'num_iterations': 25,
    'num_leaves': 100,
    'learning_rate': 0.2,
    'feature_fraction_bynode': 0.2,
    'metric': 'mse',
    'verbose': -1
}

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

booster = lgb.train(
    params,
    train_data,
    num_boost_round=params['num_iterations'],
    valid_sets=[test_data],
    valid_names=['valid'],
)



In [ ]:
# Compute per-tree SHAP values via pred_contrib=True

X_eval = X_test
n_trees = booster.num_trees()

shap_all_k = booster.predict(
    X_eval,
    num_iteration=n_trees,
    pred_contrib=True
)

shap_prev = np.zeros_like(shap_all_k)

per_tree_shap = []

for k in range(1, p + 1):
    shap_k = booster.predict(
        X_eval,
        num_iteration=k,
        pred_contrib=True
    )
    shap_k_only = shap_k - shap_prev
    per_tree_shap.append(shap_k_only)
    shap_prev = shap_k

per_tree_shap = np.stack(per_tree_shap, axis=0)

shap_sum = per_tree_shap.sum(axis=0)




In [5]:
display(per_tree_shap)

array([[[ 4.44616031e-02,  0.00000000e+00, -9.64639798e-03, ...,
          0.00000000e+00,  0.00000000e+00,  4.66817064e+01],
        [ 4.24492770e-02,  0.00000000e+00, -9.64639798e-03, ...,
          0.00000000e+00,  0.00000000e+00,  4.66817064e+01],
        [ 3.95324046e-02,  0.00000000e+00, -9.64639798e-03, ...,
          0.00000000e+00,  0.00000000e+00,  4.66817064e+01],
        ...,
        [-6.93299533e-01,  0.00000000e+00, -6.50163692e-02, ...,
          0.00000000e+00,  0.00000000e+00,  4.66817064e+01],
        [-6.93299533e-01,  0.00000000e+00, -6.50163692e-02, ...,
          0.00000000e+00,  0.00000000e+00,  4.66817064e+01],
        [-6.93299533e-01,  0.00000000e+00, -6.50163692e-02, ...,
          0.00000000e+00,  0.00000000e+00,  4.66817064e+01]],

       [[ 2.15943824e-02,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00, -3.66864583e-09],
        [ 2.15943824e-02,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e

In [6]:
# Compute SHAP values using SHAP package
explainer = shap.TreeExplainer(booster)
shap_pkg_values = explainer.shap_values(X_eval)
expected_value = explainer.expected_value

shap_pkg_full = np.zeros_like(shap_sum)
shap_pkg_full[:, :-1] = shap_pkg_values
shap_pkg_full[:, -1] = expected_value

In [7]:
diff = shap_sum - shap_pkg_full
max_abs_diff = np.max(np.abs(diff))
mean_abs_diff = np.mean(np.abs(diff))

print("Max abs difference:", max_abs_diff)
print("Mean abs difference:", mean_abs_diff)

tol = 1e-6
if max_abs_diff < tol:
    print("PASSED — per-tree SHAP sum == SHAP package (within tolerance)")
else:
    print("MISMATCH — investigate numerical or structural issues")

Max abs difference: 8.881784197001252e-16
Mean abs difference: 2.8274981998310286e-19
PASSED — per-tree SHAP sum == SHAP package (within tolerance)
